# High-dimensional image embeddings: IMSTE and UMAP

Choose dataset, graph, optimizer, and rendering settings, then compare IMSTE with UMAP on the same standardized samples. This notebook uses multilevel initialization by default and displays both embeddings with `render_datashader_embedding` to keep dense plots readable.

The full gallery, including class-mixture graph analysis, is archived at `notebooks/archive/00_high_dim_mst_gallery_full.ipynb`.


In [60]:
%load_ext autoreload
%autoreload 2

%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "src" / "high_dim_mst_gallery.py").is_file()),
    None,
)
if repo_root is not None:
    sys.path.insert(0, str(repo_root / "notebooks" / "src"))
    sys.path.insert(0, str(repo_root))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Select parameters and fit IMSTE

Set the number of papers per dataset and tune the FAMST, IMSTE, and multilevel settings below. Set `MULTILEVEL_INIT = False` to use the estimator's regular PCA initialization instead.


In [ ]:
from high_dim_mst_gallery import run_high_dim_mst_gallery
from IPython.display import display

# Dataset size
MAX_SAMPLES = 60_000
RANDOM_STATE = 42

# IMSTE graph and optimizer
MST_METHOD = "famst"  # or "prim" for exact graph construction
N_MSTS = 16
N_EPOCHS = 1000
NEGATIVE_RATIO = 1
FAMST_MAX_NEIGHBORS = 30

# Multilevel initializer (enabled by default in this notebook)
MULTILEVEL_INIT = True
MULTILEVEL_REDUCTION_RATIO = 4
MULTILEVEL_BASE_SIZE = 1024
MULTILEVEL_EPOCHS = 10

# UMAP comparison
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_EPOCHS = 200

# Datashader display
RENDER_WIDTH = 900
RENDER_HEIGHT = 700
RENDER_MIN_ALPHA = 40
RENDER_SPREAD_PX = 1
RENDER_BACKGROUND = "white"  # Change to "black" to compare contrast.

# Load the image datasets and fit IMSTE. Sample image grids and the helper's
# scatter plots are disabled; the comparison below uses Datashader instead.
gallery = run_high_dim_mst_gallery(
    max_samples=MAX_SAMPLES,
    n_msts=N_MSTS,
    n_epochs=N_EPOCHS,
    mst_method=MST_METHOD,
    negative_ratio=NEGATIVE_RATIO,
    mst_max_neighbors=FAMST_MAX_NEIGHBORS,
    random_state=RANDOM_STATE,
    plot_sample_images=False,
    plot_embeddings=False,
    multilevel_init=MULTILEVEL_INIT,
    multilevel_reduction_ratio=MULTILEVEL_REDUCTION_RATIO,
    multilevel_base_size=MULTILEVEL_BASE_SIZE,
    multilevel_epochs=MULTILEVEL_EPOCHS,
)
display(gallery["summary"])
if gallery["load_errors"]:
    display(gallery["load_errors"])


Loading MNIST (28×28) ...
  60,000 samples × 784 features; 10 classes
Loading Fashion-MNIST (28×28) ...


## 2. Fit UMAP to the same samples

The gallery helper standardizes features before IMSTE. This cell applies the same standardization to each raw dataset before fitting UMAP, so both methods receive matching input rows and feature scaling.


In [ ]:
from time import perf_counter

import umap
from IPython.display import display
from sklearn.preprocessing import StandardScaler

UMAP_RESULTS = {}
UMAP_SECONDS = {}
for dataset_name, (raw_X, labels) in gallery["datasets"].items():
    X_standardized = StandardScaler().fit_transform(raw_X)
    started = perf_counter()
    embedding = umap.UMAP(
        n_components=2,
        n_neighbors=UMAP_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric="euclidean",
        random_state=RANDOM_STATE,
        n_epochs=UMAP_EPOCHS,
        n_jobs=1,
    ).fit_transform(X_standardized)
    UMAP_RESULTS[dataset_name] = embedding
    UMAP_SECONDS[dataset_name] = perf_counter() - started
    print(f"{dataset_name}: UMAP fit in {UMAP_SECONDS[dataset_name] / 60:.1f} min")

comparison = gallery["summary"][["dataset", "samples", "input_features", "seconds"]].rename(
    columns={"seconds": "IMSTE seconds"}
)
comparison["UMAP seconds"] = comparison["dataset"].map(UMAP_SECONDS)
display(comparison)


## 3. Compare the embeddings with Datashader

Each panel rasterizes all points at the selected image size. Adjust the rendering settings above to change the mark spread, opacity, resolution, or background.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from imste.visualization import render_datashader_embedding

DATASHADER_IMAGES = {}
for dataset_name, (_, labels) in gallery["datasets"].items():
    embeddings = {
        "IMSTE · multilevel start" if MULTILEVEL_INIT else "IMSTE · PCA start":
            gallery["embeddings"][dataset_name],
        "UMAP": UMAP_RESULTS[dataset_name],
    }
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    for ax, (method, embedding) in zip(axes, embeddings.items()):
        image = render_datashader_embedding(
            embedding,
            labels,
            width=RENDER_WIDTH,
            height=RENDER_HEIGHT,
            min_alpha=RENDER_MIN_ALPHA,
            background=RENDER_BACKGROUND,
            spread_px=RENDER_SPREAD_PX,
        )
        DATASHADER_IMAGES[(dataset_name, method)] = image
        ax.imshow(image.to_pil())
        ax.set_title(f"{dataset_name} — {method}")
        ax.set_axis_off()
    display(fig)
    plt.close(fig)
